# Optymalizacja bezpośrednia: metoda GPS


Wykorzystaj poniższą metodę `generalized_pattern_search` do optymalizacji funkcji Rosenbrocka. Przetestuj następujące zestawy kierunków:

* `D=[[0, 1], [1, 0], [-1, -1]]`
* `D=[[0, -1], [-1, 0], [1, 1]]`
* `D=[[0, 1], [1, 0], [-1, 0], [0, -1]]`

oraz dwa inne samodzielnie wybrane (proszę pamiętać, że mają dodatnio rozpinać $\mathbb{R}^2$). Wykorzystaj przygotowany wcześniej kod do ewaluacji metod przeszukiwania aby określić na ile dobre są różne zestawy kierunków. Przyjmij $\alpha=1$ oraz $\epsilon=10^{-8}$.

In [1]:
function generalized_pattern_search(f, x, α, D, ε, γ=0.5)
    y, n = f(x), length(x)
    while α > ε
        improved = false
        for (i,d) in enumerate(D)
            x′ = x + α*d
            y′ = f(x′)
            if y′ < y
                x, y, improved = x′, y′, true
                D = pushfirst!(deleteat!(D, i), d)
                break
            end
        end
        if !improved
            α *= γ
        end
    end
    return x
end

generalized_pattern_search (generic function with 2 methods)

In [9]:
function f_rosenbrock(x)
    result = 0.0
    for i in 1:2:length(x)
        result += (1.0 - x[i])^2 + 100.0 * (x[i + 1] - x[i]^2)^2
    end
    return result
end

function g_rosenbrock(storage, x)
    for i in 1:2:length(x)
        storage[i] = -2.0 * (1.0 - x[i]) - 400.0 * (x[i + 1] - x[i]^2) * x[i]
        storage[i + 1] = 200.0 * (x[i + 1] - x[i]^2)
    end
    return storage
end

g_rosenbrock (generic function with 1 method)

Porównaj zbieżność z metodą największego spadku.

Spróbuj zoptymalizować metodę `generalized_pattern_search` tak, aby wykonywała jak najmniej alokacji w trakcie działania. Możesz wypróbować:
1. Prealokację `x′` przed pętlą oraz broadcasting w `x′ = x + α*d`.
2. Przepisanie `D = pushfirst!(deleteat!(D, i), d)`.
3. Bibliotekę `StaticArrays.jl`.


In [3]:
using BenchmarkTools

x0 = [0.0, 0.0]
D = [[0, 1], [1, 0], [-1, -1]]
@benchmark generalized_pattern_search($f_rosenbrock, $x0, 1.0, $D, 10^-8)

BenchmarkTools.Trial: 3278 samples with 1 evaluation per sample.
 Range (min … max):  955.000 μs … 11.719 ms  ┊ GC (min … max):  0.00% … 75.71%
 Time  (median):       1.128 ms              ┊ GC (median):     0.00%
 Time  (mean ± σ):     1.512 ms ±  1.065 ms  ┊ GC (mean ± σ):  19.43% ± 20.32%

  █▇▆▅▅▄▃▃▃▂▂▁                                                 ▁
  █████████████▇▇▇▆▃▅▃▃▁▃▄▇▇▇▇▇▇█▇▆▆▇▆▆▇▅▆▆▆▇▆▆▆▇▅▅▆▆▆▆▆▅▄▆▅▆▆ █
  955 μs        Histogram: log(frequency) by time      6.12 ms <

 Memory estimate: 2.86 MiB, allocs estimate: 75100.

Metoda największego spadku

In [14]:
using LinearAlgebra
using Plots
using Random

In [41]:
function steepest_gradient_descent(cost, grad, x0, α, γ; max_iter=1000, tol=1e-8)
    θ = copy(x0)
    f_values = []
    storage = zeros(length(θ))
    for i in 1:max_iter
        value_start = cost(θ)
        storage = grad(storage, θ)
        norm_storage = norm(storage)
        if norm_storage == 0
            break
        end
        θ_new = θ - storage .* (α / norm_storage)
        value_stop = cost(θ_new)
        push!(f_values, value_start)
        if abs(value_stop - value_start) < tol
            break
        else
            θ = θ_new
            α *= γ
        end
    end
    return θ, f_values
end

steepest_gradient_descent (generic function with 2 methods)

In [36]:
@benchmark steepest_gradient_descent($f_rosenbrock, $g_rosenbrock, $x0, 0.1, 0.99)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   55.500 μs …  23.715 ms  ┊ GC (min … max):  0.00% … 99.22%
 Time  (median):      68.800 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   108.832 μs ± 532.267 μs  ┊ GC (mean ± σ):  21.44% ±  4.91%

  ▄█▃                                                            
  ███▇▅▄▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  55.5 μs          Histogram: frequency by time          254 μs <

 Memory estimate: 188.17 KiB, allocs estimate: 4847.

In [37]:
x_min = generalized_pattern_search(f_rosenbrock, x0, 1.0, D, 10^-8)
x_min

2-element Vector{Float64}:
 0.9999970197677612
 0.9999940395355225

In [49]:
x_min2, history = steepest_gradient_descent(f_rosenbrock, g_rosenbrock, x0, 0.3, 0.99)
x_min2

2-element Vector{Float64}:
 0.9111627739796326
 0.829842718732199